# Correcting for spin contamination in zero-field splitting calculations

The accuracy of the computed zero-field splitting (ZFS) tensor can be affected by spin contamination in spin-polarized density functional theory (DFT) calculations, where the Kohn–Sham states in the two spin channels have different energies and spatial distributions. [Biktagirov et al.](https://doi.org/10.1103/PhysRevResearch.2.022024) proposed a correction scheme that effectively cancels spin contamination errors, providing more accurate computation of ZFS tensors while reducing the dependence on the exchange-correlation functional employed in DFT calculations.

In this tutorial, we demonstrate how to apply this correction to the negatively charged nitrogen-vacancy (NV$^-$) center in diamond.

In [1]:
%cd diamond_nv_qe_hdf5_correction_data

/Users/vwzyu/Soft/pyzfs/doc/tutorials/diamond_nv_qe_hdf5_correction_data


## 1 Compute magnetic dipolar interaction for triplet spin state

First, we perform a regular PyZFS calculation for the triplet spin state ($S=1$) of the NV$^-$ center in diamond, obtaining the uncorrected spin-spin ZFS tensor $\mathbf{D}^{\mathrm{SS}} = \frac{\mathbf{d}}{S(S-1/2)}$. The magnetic dipolar interaction $\mathbf{d}$ is a 3 $\times$ 3 tensor defined as

$d_{ab} = \frac{\alpha^2}{8} \sum_{ij} \chi_{ij} \int \frac{|\mathbf{r} - \mathbf{r}'|^2 \delta_{ab} - 3(\mathbf{r} - \mathbf{r}')_a(\mathbf{r} - \mathbf{r}')_b}{|\mathbf{r} - \mathbf{r}'|^5} \times [n_{ii}(\mathbf{r}) n_{jj}^*(\mathbf{r}') - n_{ij}(\mathbf{r}) n_{ij}^*(\mathbf{r}')] d\mathbf{r}d\mathbf{r}'$

where $a, b = x, y, z$, $\alpha$ is the fine-structure constant, $i, j$ are indices of occupied Kohn-Sham states $\psi(\mathbf{r})$, $n_{ij}(\mathbf{r}) = \psi_i(\mathbf{r}) \psi_j(\mathbf{r})$, and $\chi_{ij} = 1$ when $i$ and $j$ belong to the same spin channel, and $\chi_{ij} = -1$ otherwise. The $\mathbf{d}$ tensor will be used to carry out the correction for spin contamination errors.

We run the `pw.x` code of Quantum ESPRESSO for the DFT calculation.

In [2]:
%%bash
cat pw_triplet.in

mpirun -n 4 pw.x -i pw_triplet.in > pw_triplet.out

&CONTROL
calculation = 'scf'
pseudo_dir = './'
prefix = 'triplet'
/
&SYSTEM
ibrav = 0
ntyp = 2
nat = 215
tot_charge = -1.0
tot_magnetization = 2.0
nspin = 2
nbnd = 628
ecutwfc = 60.0
/
&ELECTRONS
diago_full_acc = .true.
/
ATOMIC_SPECIES
C  12.01099968  C_ONCV_PBE-1.2.upf
N  14.00699997  N_ONCV_PBE-1.2.upf
CELL_PARAMETERS angstrom
10.704  0.000  0.000
 0.000 10.704  0.000
 0.000  0.000 10.704
K_POINTS gamma
ATOMIC_POSITIONS crystal
N 0.09152860 0.09152860 0.09152860
C 0.00406095 0.16656435 0.16656435
C 0.08375956 0.25132100 0.25132100
C 0.16656435 0.00406095 0.16656435
C 0.25132100 0.08375956 0.25132100
C 0.16656435 0.16656435 0.00406095
C 0.25132100 0.25132100 0.08375956
C 0.00008479 0.00008479 0.33283360
C 0.08339786 0.08339786 0.41672965
C 0.00001566 0.16656881 0.50004871
C 0.08326671 0.24989560 0.58351323
C 0.16656881 0.00001566 0.50004871
C 0.24989560 0.08326671 0.58351323
C 0.16713153 0.16713153 0.33408428
C 0.25020487 0.25020487 0.41712458
C 0.99984338 0.99984338 0.66702816
C 0.0

Next, we run PyZFS to compute the ZFS tensor.

In [ ]:
%%bash
mpirun -n 4 pyzfs --wfcfmt qeh5 --prefix triplet --memory low > zfs_triplet.out
mv zfs.xml zfs_triplet.xml

From the XML output of PyZFS, we extract the uncorrected ZFS parameters D and E, as well as the magnetic dipolar interaction $\mathbf{d}$ tensor.

In [3]:
import numpy as np
import xml.etree.ElementTree as ET

tree = ET.parse("zfs_triplet.xml")
root = tree.getroot()

ZFS_D = float(root.find("D").text)
ZFS_E = float(root.find("E").text)

print(f"D = {ZFS_D} MHz")
print(f"E = {ZFS_E} MHz")

D_triplet = root.find("DTensor").text
D_triplet = D_triplet.replace("[", "").replace("]", "").replace("\n", " ")
D_triplet = np.fromstring(D_triplet, sep=" ").reshape(3, 3)

# PyZFS internally assumes S = 1, so d = D / 2
d_triplet = D_triplet / 2

print("d tensor:")
print(d_triplet)

D = 3033.97 MHz
E = -0.68 MHz
d tensor:
[[-1.37407286e-01  5.05645425e+02  5.05897945e+02]
 [ 5.05645425e+02  1.36650451e-01  5.05442520e+02]
 [ 5.05897945e+02  5.05442520e+02  7.56833945e-04]]


## 2 Compute magnetic dipolar interaction for three singlet spin states

To correct for spin contamination in the spin-spin ZFS tensor, we compute

$\tilde{\mathbf{D}}^{\mathrm{SS}} = \frac{\mathbf{d}_{(m_S=S)} - \bar{\mathbf{d}}_{(m_S=S-1)}}{2S-1}$

where $\bar{\mathbf{d}}_{(m_S=S-1)}$ is the average $\mathbf{d}_{(m_S=S−1)}$ among all the possible $m_S=S−1$ configurations. For our target triplet state ($S=1$), we compute and average $\mathbf{d}$ for the singlet states ($S=0$).

The unpaired electrons of the NV$^-$ center in diamond are
equally distributed among the three carbon dangling bonds
surrounding the vacancy. In an $m_S=0$ configuration, the spin up electron localizes at one of the three dangling bonds, while the spin down electron is shared by the other two adjacent carbon atoms, resulting in three configurations of singlet states.

To simulate these singlet states, we change the `tot_magnetization` input keyword of `pw.x` from 2.0 to 0.0, and we initialize the magnetization by setting the `starting_magnetization` such that the spin up electron localizes at one of the three carbon atoms around the vacancy (for example, `C1` in the input file below), while the spin down electron is shared by the other two carbon atoms around the vacancy (`C2` and `C3`).

In [4]:
%%bash
cat pw_singlet_1.in

mpirun -n 4 pw.x -i pw_singlet_1.in > pw_singlet_1.out

&CONTROL
calculation = 'scf'
pseudo_dir = './'
prefix = 'singlet_1'
/
&SYSTEM
ibrav = 0
ntyp = 5
nat = 215
tot_charge = -1.0
tot_magnetization = 0.0
starting_magnetization(1) =  0.5  ! C1
starting_magnetization(2) = -0.25 ! C2
starting_magnetization(3) = -0.25 ! C3
starting_magnetization(4) =  0.0  ! C
starting_magnetization(5) =  0.0  ! N
nspin = 2
nbnd = 628
ecutwfc = 60.0
/
&ELECTRONS
diago_full_acc = .true.
mixing_beta = 0.2
/
ATOMIC_SPECIES
C1  12.01099968  C_ONCV_PBE-1.2.upf
C2  12.01099968  C_ONCV_PBE-1.2.upf
C3  12.01099968  C_ONCV_PBE-1.2.upf
C   12.01099968  C_ONCV_PBE-1.2.upf
N   14.00699997  N_ONCV_PBE-1.2.upf
CELL_PARAMETERS angstrom
10.704  0.000  0.000
 0.000 10.704  0.000
 0.000  0.000 10.704
K_POINTS gamma
ATOMIC_POSITIONS crystal
N 0.09152860 0.09152860 0.09152860
C 0.00406095 0.16656435 0.16656435
C 0.08375956 0.25132100 0.25132100
C 0.16656435 0.00406095 0.16656435
C 0.25132100 0.08375956 0.25132100
C 0.16656435 0.16656435 0.00406095
C 0.25132100 0.25132100 0.083759

Next, we run PyZFS to compute the $\mathbf{d}$ tensor.

In [ ]:
%%bash
mpirun -n 4 pyzfs --wfcfmt qeh5 --prefix singlet_1 --memory low > zfs_singlet_1.out
mv zfs.xml zfs_singlet_1.xml

The same is done for the other two singlet configurations that have the spin up electron localized on `C2` or `C3`.

In [5]:
%%bash
cat pw_singlet_2.in

mpirun -n 4 pw.x -i pw_singlet_2.in > pw_singlet_2.out
mpirun -n 4 pyzfs --wfcfmt qeh5 --prefix singlet_2 --memory low > zfs_singlet_2.out
mv zfs.xml zfs_singlet_2.xml

&CONTROL
calculation = 'scf'
pseudo_dir = './'
prefix = 'singlet_2'
/
&SYSTEM
ibrav = 0
ntyp = 5
nat = 215
tot_charge = -1.0
tot_magnetization = 0.0
starting_magnetization(1) = -0.125 ! C1
starting_magnetization(2) =  0.25  ! C2
starting_magnetization(3) = -0.125 ! C3
starting_magnetization(4) =  0.0   ! C
starting_magnetization(5) =  0.0   ! N
nspin = 2
nbnd = 628
ecutwfc = 60.0
/
&ELECTRONS
diago_full_acc = .true.
mixing_beta = 0.2
/
ATOMIC_SPECIES
C1  12.01099968  C_ONCV_PBE-1.2.upf
C2  12.01099968  C_ONCV_PBE-1.2.upf
C3  12.01099968  C_ONCV_PBE-1.2.upf
C   12.01099968  C_ONCV_PBE-1.2.upf
N   14.00699997  N_ONCV_PBE-1.2.upf
CELL_PARAMETERS angstrom
10.704  0.000  0.000
 0.000 10.704  0.000
 0.000  0.000 10.704
K_POINTS gamma
ATOMIC_POSITIONS crystal
N 0.09152860 0.09152860 0.09152860
C 0.00406095 0.16656435 0.16656435
C 0.08375956 0.25132100 0.25132100
C 0.16656435 0.00406095 0.16656435
C 0.25132100 0.08375956 0.25132100
C 0.16656435 0.16656435 0.00406095
C 0.25132100 0.25132100 0.0

In [6]:
%%bash
cat pw_singlet_3.in

mpirun -n 4 pw.x -i pw_singlet_3.in > pw_singlet_3.out
mpirun -n 4 pyzfs --wfcfmt qeh5 --prefix singlet_3 --memory low > zfs_singlet_3.out
mv zfs.xml zfs_singlet_3.xml

&CONTROL
calculation = 'scf'
pseudo_dir = './'
prefix = 'singlet_3'
/
&SYSTEM
ibrav = 0
ntyp = 5
nat = 215
tot_charge = -1.0
tot_magnetization = 0.0
starting_magnetization(1) = -0.125 ! C1
starting_magnetization(2) = -0.125 ! C2
starting_magnetization(3) =  0.25  ! C3
starting_magnetization(4) =  0.0   ! C
starting_magnetization(5) =  0.0   ! N
nspin = 2
nbnd = 628
ecutwfc = 60.0
/
&ELECTRONS
diago_full_acc = .true.
mixing_beta = 0.2
/
ATOMIC_SPECIES
C1  12.01099968  C_ONCV_PBE-1.2.upf
C2  12.01099968  C_ONCV_PBE-1.2.upf
C3  12.01099968  C_ONCV_PBE-1.2.upf
C   12.01099968  C_ONCV_PBE-1.2.upf
N   14.00699997  N_ONCV_PBE-1.2.upf
CELL_PARAMETERS angstrom
10.704  0.000  0.000
 0.000 10.704  0.000
 0.000  0.000 10.704
K_POINTS gamma
ATOMIC_POSITIONS crystal
N 0.09152860 0.09152860 0.09152860
C 0.00406095 0.16656435 0.16656435
C 0.08375956 0.25132100 0.25132100
C 0.16656435 0.00406095 0.16656435
C 0.25132100 0.08375956 0.25132100
C 0.16656435 0.16656435 0.00406095
C 0.25132100 0.25132100 0.0

From the XML outputs of PyZFS, we extract the magnetic dipolar interaction $\mathbf{d}$ tensors of the singlet states.

In [7]:
d_singlet = {}

for conf in range(1, 4):
    tree = ET.parse(f"zfs_singlet_{conf}.xml")
    root = tree.getroot()

    D_singlet = root.find("DTensor").text
    D_singlet = D_singlet.replace("[", "").replace("]", "").replace("\n", " ")
    D_singlet = np.fromstring(D_singlet, sep=" ").reshape(3, 3)

    # PyZFS internally assumes S = 1, so d = D / 2
    d_singlet[conf] = D_singlet / 2

## 3 Compute zero-field splitting with correction for spin contamination

Finally we build and diagonalize the spin decontaminated spin-spin ZFS tensor $\tilde{\mathbf{D}}^{\mathrm{SS}}$.

In [8]:
D_corrected = d_triplet - (d_singlet[1] + d_singlet[2] + d_singlet[3]) / 3

ev, evc = np.linalg.eig(D_corrected)

# For triplet states, compute D and E parameters:
# Denote three eigenvalues as Dx, Dy, Dz: |Dz| > |Dx| > |Dy|
# D = 3/2 Dz, E = 1/2(Dx - Dy)
args = np.abs(ev).argsort()
dy, dx, dz = ev[args]
ZFS_D_corrected = 1.5 * dz
ZFS_E_corrected = 0.5 * (dx - dy)

# Summarize results
print("Uncorrected ZFS:")
print(f"D = {ZFS_D:.2f}")
print(f"E = {ZFS_E:.2f}")
print()
print("Corrected ZFS:")
print(f"D = {ZFS_D_corrected:.2f}")
print(f"E = {ZFS_E_corrected:.2f}")

Uncorrected ZFS:
D = 3033.97
E = -0.68

Corrected ZFS:
D = 2544.28
E = -0.59
